In [ ]:
import requests
import time
from config import Config

def route_query(query, chat_history=None):
    url = f"https://clovastudio.stream.ntruss.com/testapp/v1/routers/4cdpojoi/versions/1/route"
    #url = Config.ROUTER_API
    headers = {
        "Authorization": f"Bearer {Config.API_KEY}",
        "X-NCP-CLOVASTUDIO-REQUEST-ID": Config.REQUEST_ID_ROUTER,
        "Content-Type": "application/json"
    }
    data = {"query": query}
    if chat_history:
        data["chatHistory"] = chat_history

    while True:
        response = requests.post(url, headers=headers, json=data)
        if response.status_code == 429:
            time.sleep(5)
            continue
        return response.json()

In [42]:
#라우터 성능 확인하기 위해 1차 확인!

query1= "선릉에 유명한 피잣집을 알려줘"
router_result1 = route_query(query1)

query2= "주식을 잘 하려면 무엇부터 공부해야할까?"
router_result2 = route_query(query2)

# JSON 전체 보기
print(router_result1)
print(router_result2)

{'status': {'code': '20000', 'message': 'OK'}, 'result': {'domain': {'result': '지역 검색', 'called': True}, 'blockedContent': {'result': ['NoOnlyone'], 'called': True}, 'safety': {'result': [], 'called': False}, 'usage': {'promptTokens': 636, 'completionTokens': 41, 'totalTokens': 677}}}
{'status': {'code': '20000', 'message': 'OK'}, 'result': {'domain': {'result': '', 'called': True}, 'blockedContent': {'result': [], 'called': False}, 'safety': {'result': [], 'called': False}, 'usage': {'promptTokens': 639, 'completionTokens': 43, 'totalTokens': 682}}}


In [ ]:
def get_chat_response(query, chat_history=None):
    # 먼저 라우터를 통해 도메인/차단 판단
    router_result = route_query(query, chat_history)

    # 목적 외 사용 판단 예시 
    domain = router_result.get("result", {}).get("domain", {}).get("result", "")

    # 차단 도메인 목록 정의
    AllOW_DOMAINS = ["역사 질문"]

    if domain not in AllOW_DOMAINS:
        return {
            "message" : "역사 관련 외 질문은 답변이 불가능합니다.",
            "filtered_domain" : domain or "미분류"
        }
    # 라우터에 걸리지 않았다면 정상적으로로 HCX 사용
    url = Config.CHAT_COMPLETIONS_API
    headers = {
        'Authorization': f'Bearer {Config.API_KEY}',
        'X-NCP-CLOVASTUDIO-REQUEST-ID': Config.REQUEST_ID_CHAT,
        'Content-Type': 'application/json',
    }

    system_prompt = "당신은 업무 도우미 입니다."
    messages = [{'role': 'system', 'content': system_prompt}]

    if chat_history:
        messages.extend(chat_history[-3:])
    else:
        messages.append({'role': 'user', 'content': query})

    data = {
        'messages': messages,
        "maxTokens": 512,
        "seed": 0,
        "temperature": 0.4,
        "topP": 0.4,
        "topK": 0,
        "repeatPenalty": 5.0
    }

    response = requests.post(url, headers=headers, json=data)
    return response.json()

In [19]:
response = get_chat_response("선릉에 삼겹살 집을 알려줘줘")

print(response)

{'message': '업무 외 질문이므로 해당 질문에 대해선 답변이 불가능합니다.', 'filtered_domain': '지역 검색'}


In [17]:
response = get_chat_response("삼성전자 주가는는 한달 전에 비해 어때?")

print(response)

{'message': '업무 외 질문이므로 해당 질문에 대해선 답변이 불가능합니다.', 'filtered_domain': '주식'}
